In [0]:


-- 1.  合并新数据到DIM表（UPSERT）
MERGE INTO gold_dim_user_scd2 AS target
USING (select * from silver_user_info where is_latest = true) AS source
ON target.id = source.id AND target.is_latest = true
    WHEN MATCHED AND (
    MD5(CONCAT(target.id, target.name, target.phone_num, target.email,target.user_level)) <> md5(CONCAT(source.id, source.name, source.phone_num, source.email,source.user_level))
) THEN
        UPDATE SET                
                target.is_latest = false,
                target.valid_to = DATE_SUB(source.update_timestamp, 1),--current_timestamp() DATE_SUB(CURRENT_DATE(), 1)  应该用SILVERscd2表的update_timestamp最准确
                target.update_time = current_timestamp()
    WHEN NOT MATCHED THEN
        INSERT (id          ,
                login_name            ,
                nick_name             ,
                passwd                ,
                name                  ,
                phone_num             ,
                email                 ,
                head_img              ,
                user_level            ,
                birthday              ,
                gender                ,
                create_time           ,
                operate_time          ,
                status                ,
                --source_file           ,
                --load_timestamp        ,
                valid_from    ,
                valid_to,
                is_latest, 
                load_time,
                update_time
                )
        VALUES (source.id          ,
                source.login_name            ,
                source.nick_name             ,
                source.passwd                ,
                source.name                  ,
                source.phone_num             ,
                source.email                 ,
                source.head_img              ,
                source.user_level            ,
                source.birthday              ,
                source.gender                ,
                source.create_time           ,
                source.operate_time          ,
                source.status                ,
                -- source.source_file           , 
                source.update_timestamp        ,
                CAST('9999-12-31' AS DATE),
                true                        ,
                current_timestamp()        ,
                current_timestamp() 
                );